# Stance Fine-Tuning Hyperparameter Playground (Colab, free T4)

**Purpose:** run the *exact same* hyperparameters as `finetune_stance.job` on the
full training dataset, but interactively on a free Colab T4 -- so you can watch the
loss curve live and stop early the moment it looks reasonable, instead of finding
out only after a full Snellius job finishes.

Every training hyperparameter here matches `finetune_stance.job` exactly (dataset,
batch size, gradient accumulation, epochs, packing, weight decay, warmup, LoRA
config). The one thing that *can't* match: `bf16` requires Ampere+ GPU hardware
(A100) -- a free T4 doesn't support it, so this notebook auto-falls-back to `fp16`.
That's a hardware limit, not a setting.

**Beyond just a final loss number, this notebook shows you:**
- the loss curve over training steps (updates live in the training table too, but a
  plotted curve makes the trend easier to read at a glance)
- confirmation that LoRA weights *actually changed* (sum-of-abs-diff check, not a
  loose `np.allclose` that can miss LoRA A's tiny Gaussian-init updates)
- **before/after predictions on real held-out test examples** -- so you can
  directly see the model's output change, not just infer it from a number

**You can interrupt training early:** once the loss curve looks like it's trending
down and you're satisfied, hit the cell's stop button (or Runtime -> Interrupt
execution). The model keeps whatever weights it had at that point, and every cell
after the training cell still works against that partially-trained state.

**Setup:** Colab menu -> Runtime -> Change runtime type -> **T4 GPU** (free tier).


## 1. Install dependencies

Same pinned versions Unsloth's own example notebooks use on Colab.


In [ ]:
%%capture
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


## 2. Load the base model + tokenizer

Same checkpoint as `finetune_stance.job`: `unsloth/Qwen3-1.7B-unsloth-bnb-4bit`.


In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"  # same as finetune_stance.job
MAX_SEQ_LENGTH = 512  # same as --max_length in finetune_stance.job

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto: fp16 on this T4, bf16 on Snellius's A100
    load_in_4bit=True,
)


## 3. Apply LoRA

Identical to `apply_lora()` in `finetune_stance.py`. **This is the cell to edit**
if you want to see what changing `r` / `lora_alpha` / `lora_dropout` does to the
loss curve below -- everything else in this notebook stays the same.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ_LENGTH,
    random_state=47,
)


## 4. Load the full training dataset

Same `conversations` format + same broadened prompt as the production pipeline
(see notebooks 01/02 for the full walkthrough of that format). Colab has no access
to your Mac's local files, so this pulls straight from the HF Hub.

Loading the **full** dataset (not a subsample) -- matching `finetune_stance.job`'s
`--data_dir` exactly, so the step count and loss trend here are representative of
the real run, not an artificially small preview.


In [ ]:
DATASET_ID = "nityaak/semeval-stance-conversations"  # same as finetune_stance.job's --data_dir

from datasets import load_dataset

dataset = load_dataset(DATASET_ID, split="train")
print(f"Using {len(dataset)} examples (full dataset, no subsampling)")
dataset[0]


## 5. Split each conversation into prompt/completion

Instead of flattening the conversation into one `"text"` string, split it into
`"prompt"` (the user turn) and `"completion"` (the assistant's JSON answer). This is
what lets `completion_only_loss` (set below) restrict training loss to just the
answer -- a flattened `"text"` field loses the boundary between the fixed
instructions and the answer, so loss would get computed over both. (Verified:
Qwen3's chat template doesn't support the alternative `assistant_only_loss`
mechanism -- it returns an all-zero mask -- so this prompt/completion split is the
approach that actually works.) `SFTTrainer` applies the chat template to
prompt+completion internally, so we don't render text ourselves anymore.


In [ ]:
def split_prompt_completion(examples):
    convos = examples["conversations"]
    return {
        "prompt": [convo[:-1] for convo in convos],
        "completion": [convo[-1:] for convo in convos],
    }

dataset = dataset.map(split_prompt_completion, batched=True)
print(dataset[0]["prompt"])
print(dataset[0]["completion"])


## 6. Load real held-out test examples for a before/after check

Rather than splitting a few examples off the tiny training subsample (which would
only be "held out" from this one experiment, not a genuine independent test set),
this pulls a handful from the actual held-out gold test set -- the same one the full
Snellius pipeline evaluates against (`semeval2016_task6_testdata_gold`, confirmed
zero overlap with training data).

Run `04_prepare_test_set.ipynb` once first to push it to the Hub.


In [ ]:
TEST_DATASET_ID = "nityaak/semeval-stance-test-gold"  # from 04_prepare_test_set.ipynb
EVAL_SAMPLE_SIZE = 5

test_dataset = load_dataset(TEST_DATASET_ID, split="train")
eval_examples = test_dataset.shuffle(seed=42).select(range(EVAL_SAMPLE_SIZE))

train_examples = dataset  # train on the full subsample loaded in section 4 -- nothing held out from it anymore
print(f"Training on {len(train_examples)}, evaluating on {len(eval_examples)} real held-out test examples")


## 7. What does the model predict BEFORE fine-tuning?


In [ ]:
def generate_stance(model, tokenizer, conversation):
    FastLanguageModel.for_inference(model)
    user_turn = [conversation[0]]  # just the user prompt, not the gold answer
    prompt_text = tokenizer.apply_chat_template(
        user_turn, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=20, use_cache=True)
    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return generated.strip()


print("BEFORE fine-tuning:")
for example in eval_examples:
    convo = example["conversations"]
    predicted = generate_stance(model, tokenizer, convo)
    true_answer = convo[1]["content"]
    print(f"  predicted={predicted!r}  true={true_answer!r}")


## 8. Set hyperparameters -- identical to `finetune_stance.job`

Every value here matches the real job exactly (only `fp16`/`bf16` is auto-detected
per hardware, see intro). **Change `learning_rate` / `num_train_epochs` / etc. here
and re-run from this cell down** to see the effect on the loss curve and the
before/after predictions -- then carry over whatever you land on to
`finetune_stance.job`.


In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="colab_outputs",
    per_device_train_batch_size=8,   # same as finetune_stance.job
    gradient_accumulation_steps=2,   # same as finetune_stance.job -- effective batch size 16
    num_train_epochs=1,              # same as finetune_stance.job
    learning_rate=2e-5,              # SFTConfig's default -- try e.g. 1e-4 or 2e-4 here
    weight_decay=0.01,               # same as finetune_stance.job
    warmup_ratio=0.1,                # same as finetune_stance.job -- warmup_steps must be a plain int, this is the float-ratio field
    optim="adamw_8bit",              # same as finetune_stance.job
    fp16=not torch.cuda.is_bf16_supported(),  # T4 -> fp16; A100/H100 -> bf16 (hardware limit, can't force-match)
    bf16=torch.cuda.is_bf16_supported(),
    packing=True,                    # same as finetune_stance.job -- bundles short examples into fewer sequences,
                                      # so actual step count will be lower than len(dataset) / effective_batch_size
    logging_steps=1,                 # log every step -- we want the full loss curve, not just every 10th point
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,       # only grade the JSON answer, not the fixed instructions -- see section 5
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_examples,
)


## 9. Snapshot LoRA weights before training

So we can confirm afterward that they actually changed.


In [ ]:
def snapshot_trainable_params(model):
    return {
        name: param.detach().clone()
        for name, param in model.named_parameters()
        if param.requires_grad
    }


FastLanguageModel.for_training(model)  # undo the for_inference() switch from the BEFORE check above
before_snapshot = snapshot_trainable_params(model)


## 10. Train

Watch the loss column in the live table below. **Once it looks like it's trending
down and you're satisfied, you can interrupt this cell** (stop button / Runtime ->
Interrupt execution) instead of waiting for the full run to finish -- the
`try`/`except` below catches that cleanly so every cell after this one still works
against whatever the model learned up to that point.


In [ ]:
try:
    trainer_stats = trainer.train()
    print(trainer_stats)
except KeyboardInterrupt:
    print("Training interrupted manually -- continuing with the model as currently trained.")


## 11. Plot the loss curve


In [ ]:
import matplotlib.pyplot as plt

logged_steps = [entry for entry in trainer.state.log_history if "loss" in entry]
steps = [entry["step"] for entry in logged_steps]
losses = [entry["loss"] for entry in logged_steps]

plt.figure()
plt.plot(steps, losses, marker="o")
plt.xlabel("Step")
plt.ylabel("Training loss")
plt.title("Training loss curve")
plt.show()


## 12. Verify LoRA weights actually updated

Same check as `finetune_stance.py` -- sum of absolute differences per tensor, not a
loose-tolerance comparison (LoRA A's tiny Gaussian init can hide real changes under
`np.allclose`).


In [ ]:
def verify_weights_updated(model, before_snapshot):
    print("Verifying LoRA weight updates (sum of abs differences per tensor):")
    unchanged = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        diff = (param.detach() - before_snapshot[name]).abs().sum().item()
        print(f"  {name}: {diff:.6f}")
        if diff == 0:
            unchanged.append(name)
    if unchanged:
        print(f"WARNING: {len(unchanged)} trainable param(s) never changed: {unchanged}")
    else:
        print("All trainable parameters changed during training.")


verify_weights_updated(model, before_snapshot)


## 13. What does the model predict AFTER fine-tuning? (compare to section 7)


In [ ]:
print("AFTER fine-tuning:")
for example in eval_examples:
    convo = example["conversations"]
    predicted = generate_stance(model, tokenizer, convo)
    true_answer = convo[1]["content"]
    print(f"  predicted={predicted!r}  true={true_answer!r}")


## 14. GPU memory used (sanity check before scaling up sample size)


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_memory = round(gpu_stats.total_memory / 1024**3, 2)
print(f"Peak reserved memory: {used_memory} GB / {total_memory} GB ({gpu_stats.name})")


## 15. Full test-set evaluation: macro F1 (FAVOR/AGAINST/NONE)

Runs the fine-tuned model over the **entire** held-out test set (1249 examples, not
just the `EVAL_SAMPLE_SIZE` spot check above) and computes macro-averaged F1 across
all three classes -- a real accuracy signal instead of a tiny sample. This will take
a while on a free T4 (one generation call per example) -- progress prints every 100
examples, and you can interrupt early (same pattern as the training cell) to get an
F1 score on however many examples completed so far.


In [ ]:
import re

from sklearn.metrics import classification_report, f1_score

LABELS = ["FAVOR", "AGAINST", "NONE"]


def parse_stance(text):
    match = re.search(r'"stance"\s*:\s*"(FAVOR|AGAINST|NONE)"', text)
    return match.group(1) if match else "PARSE_ERROR"


def evaluate_full_test_set(model, tokenizer, test_dataset):
    """Runs generate_stance() over every example and returns (y_true, y_pred) lists.
    Reused for both the fine-tuned and the untrained (pre-training) model below."""
    y_true, y_pred = [], []
    try:
        for i, example in enumerate(test_dataset):
            convo = example["conversations"]
            generated = generate_stance(model, tokenizer, convo)
            y_true.append(parse_stance(convo[1]["content"]))
            y_pred.append(parse_stance(generated))
            if (i + 1) % 100 == 0:
                print(f"{i + 1}/{len(test_dataset)} done")
    except KeyboardInterrupt:
        print(f"Interrupted after {len(y_true)} examples -- scoring on what we have so far.")
    return y_true, y_pred


def plot_f1(y_true, y_pred, title):
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))
    per_class_f1 = f1_score(y_true, y_pred, labels=LABELS, average=None, zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)

    plt.figure()
    plt.bar(LABELS + ["MACRO AVG"], list(per_class_f1) + [macro_f1])
    plt.ylabel("F1 score")
    plt.ylim(0, 1)
    plt.title(f"{title} (macro = {macro_f1:.3f}, n={len(y_true)})")
    plt.show()


y_true_trained, y_pred_trained = evaluate_full_test_set(model, tokenizer, test_dataset)
print(f"\nEvaluated on {len(y_true_trained)} examples\n")


In [ ]:
plot_f1(y_true_trained, y_pred_trained, "Test-set F1 -- fine-tuned model")


## 16. Push the fine-tuned model to the Hub

Merges the LoRA adapter into the base model and pushes it -- same mechanism as
`push_to_hub_merged()` in `finetune_stance.py`. Uses a **separate repo id** from the
production Snellius run so a quick Colab experiment never overwrites your official
model. Pushing always requires authentication (unlike just reading a public
dataset), even for a private repo.


In [ ]:
from huggingface_hub import login

login()  # paste a token with write access from https://huggingface.co/settings/tokens -- only needed once per Colab runtime

PUSH_TO_HUB_ID = "nityaak/qwen3-1.7b-stance-semeval-qlora-colab-sandbox"
PRIVATE = True

# Uncomment when ready:
# model.push_to_hub_merged(PUSH_TO_HUB_ID, tokenizer, save_method="merged_16bit", private=PRIVATE)
# print(f"Pushed to https://huggingface.co/{PUSH_TO_HUB_ID}")


## 17. F1 for the UNTRAINED model, on the same full test set

Reloads a fresh copy of the base model + LoRA setup (same as sections 2-3), instead
of reverting the trained model's weights in place -- simpler, and doesn't touch or
risk the fine-tuned `model`/`tokenizer` at all. Uses separate variable names
(`base_model`/`base_tokenizer`) so nothing from section 16 onward is affected.


In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
base_model = FastLanguageModel.get_peft_model(
    base_model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ_LENGTH,
    random_state=47,
)

y_true_untrained, y_pred_untrained = evaluate_full_test_set(base_model, base_tokenizer, test_dataset)
print(f"\nEvaluated on {len(y_true_untrained)} examples\n")


In [ ]:
plot_f1(y_true_untrained, y_pred_untrained, "Test-set F1 -- zero-shot (before fine-tuning)")


## 18. Generalization check: F1 on mtcsd (out-of-domain test set)

The model was only trained on semeval. This checks how well it generalizes to a
completely different domain set (mtcsd: Biden/Bitcoin/SpaceX/Tesla/Trump) it's never
seen -- using mtcsd's own held-out **test** split (2,371 examples, not the training
rows notebook 02 used), pushed via `05_prepare_additional_test_sets.ipynb`.

Uses a sample (`MTCSD_SAMPLE_SIZE`) to keep runtime reasonable on a free T4 --
increase it (up to 2371) for a fully rigorous number. Uses the fine-tuned
`model`/`tokenizer` from section 8-10, not `base_model`.


In [ ]:
MTCSD_DATASET_ID = "nityaak/mtcsd-stance-test"  # from 05_prepare_additional_test_sets.ipynb
MTCSD_SAMPLE_SIZE = 300

mtcsd_test_dataset = load_dataset(MTCSD_DATASET_ID, split="train")
mtcsd_test_dataset = mtcsd_test_dataset.shuffle(seed=42).select(
    range(min(MTCSD_SAMPLE_SIZE, len(mtcsd_test_dataset)))
)

y_true_mtcsd, y_pred_mtcsd = evaluate_full_test_set(model, tokenizer, mtcsd_test_dataset)
print(f"\nEvaluated on {len(y_true_mtcsd)} examples\n")


In [ ]:
plot_f1(y_true_mtcsd, y_pred_mtcsd, "mtcsd test F1 -- semeval-trained model (out-of-domain)")


## 19. Generalization check: F1 on EZ-STANCE (out-of-domain test set)

Same idea as section 18, using EZ-STANCE's mixed test split (7,798 examples --
`ezstance_test_mixed.csv`, the one already referenced in your existing Tilburg eval
outputs) pushed via `05_prepare_additional_test_sets.ipynb`.


In [ ]:
EZSTANCE_DATASET_ID = "nityaak/ezstance-stance-test-mixed"  # from 05_prepare_additional_test_sets.ipynb
EZSTANCE_SAMPLE_SIZE = 300

ezstance_test_dataset = load_dataset(EZSTANCE_DATASET_ID, split="train")
ezstance_test_dataset = ezstance_test_dataset.shuffle(seed=42).select(
    range(min(EZSTANCE_SAMPLE_SIZE, len(ezstance_test_dataset)))
)

y_true_ezstance, y_pred_ezstance = evaluate_full_test_set(model, tokenizer, ezstance_test_dataset)
print(f"\nEvaluated on {len(y_true_ezstance)} examples\n")


In [ ]:
plot_f1(y_true_ezstance, y_pred_ezstance, "ezstance test F1 -- semeval-trained model (out-of-domain)")


## Recap

- Full dataset + identical hyperparameters to `finetune_stance.job` (only
  `fp16`/`bf16` differs, and only because of T4 vs A100 hardware) -- this is a
  faithful preview of the real run, not a shrunk-down approximation
- You can interrupt training early once the loss curve looks reasonable -- the
  model, loss history, and weight snapshot all stay usable for the cells below
- Nothing here touches the Hub -- this notebook never modifies your production
  dataset/model repos
- Once you're happy with a hyperparameter change: update `finetune_stance.job`'s
  flags to match and run the full dataset on Snellius via `sbatch finetune_stance.job`
